# **TUTORIAL** 5m: Assess absolute environmental sustainability for monitoring

Monitoring assesses whether user-defined circular economy (CE) transition
pathways achieve absolute environmental sustainability. CE performance levels
are set first, then dynamic MFA and time-explicit prospective LCA calculate
the resulting material flows and environmental burdens. This notebook completes
the assessment by comparing those burdens with allocated carrying capacities
through the Absolute Sustainability Ratio (ASR), using `run_asr(...)`.

## Before starting...

Complete Tutorials 1--4 for the foregrounds and backgrounds you want to assess.
Calculate allocated carrying capacities following the
[optional pyaesa tutorial](../optional_workflows/pyaesa_allocated_carrying_capacity.ipynb),
or supply your own values based on your chosen allocation method and shares.
The environmental burdens and allocated carrying capacities must refer to the
same activity, functional unit, and system boundaries.

The steps below select the completed LCA results and allocated carrying capacity
inputs, calculate ASRs, and read the annual and cumulative assessments.
Target setting follows [Tutorial 5o](5o_run_optimization.ipynb).


In [ ]:
import json
from pathlib import Path

import pandas as pd

from pytoce import run_asr, set_workspace

## Select the workspace

Run `set_workspace(...)` before any later function call. It sets the active
workspace root for the current Python session. Use the workspace containing
your completed LCA results and allocated carrying capacity inputs.

As in the preceding tutorials, this example assesses *2019frozen_SSP2demand*
using the scenario name `frozen_2019_default_baseline`. Select another
completed foreground to assess a different CE performance trajectory.


In [ ]:
workspace = Path.home() / "Documents" / "pytoce_workspace"
set_workspace(workspace)
foreground = "frozen_2019_default_baseline"

## Select allocated carrying capacities

The example uses `pyaesa_plastics`, written by the optional pyaesa tutorial under
`data_raw/allocated_cc/results/`. Its YAML identifies the annual and cumulative
CSV files. The annual table covers all 12 regions, 2019--2100, and the 16 EF 3.1
impact categories. The cumulative table supplies a GWP100 allocated carrying
capacity for 2019--2100 for each region.

Replace the name below if you supplied another result. The optional tutorial
uses `p5`, `p50`, and `p95`: the 5th, 50th (median), and 95th percentiles of
the allocated carrying capacity distributions. Each percentile is used as a
separate deterministic input to the ASR calculation. `allocated_cc_case=None`
selects every available case.

The optional tutorial uses steady-state carrying capacities for annual assessment
and a dynamic climate change carrying capacity for cumulative GWP100 over
2019--2100.


In [ ]:
allocated_cc_result = "pyaesa_plastics"
asr_report = run_asr(
    foreground_scenario=foreground,
    allocated_cc_result=allocated_cc_result,
)
asr_report.as_dict()

The default selects the four REMIND SSP2 background scenarios described in
[Tutorial 2](2_prepare_backgrounds.ipynb). It reads completed LCA results
containing both EF 3.1 and PB-LCIA, as calculated in Tutorial 4. ASRs use the
16 EF 3.1 impact categories. If you calculated LCA with `lcia_method="ef31"`,
pass the same selection to `run_asr(...)`.

Select a background with `background_scenario="remind_ssp2_npi"`, or pass a
sequence of names. Each foreground, background, and allocated carrying capacity
case is assessed separately. Results are written under
`results/<foreground>/asr_results/`.


## Read annual ASRs

Annual ASRs divide environmental burdens by the allocated carrying capacities
for the same region, impact category, and year. An ASR <= 1 indicates compliance
with the allocated carrying capacities, whereas an ASR > 1 indicates
transgression. ASRs are dimensionless.

Compliance with annual allocated carrying capacities is assessed over
2020--2100. The 2019 rows are included for reporting and have
`assessed_for_compliance=False`.

The next cell reads the EUR climate change assessment in 2050 for each selected
background and allocated carrying capacity case.


In [ ]:
annual = pd.concat(
    [pd.read_parquet(item.annual_asr) for item in asr_report.results],
    ignore_index=True,
)
annual.loc[
    annual["region"].eq("EUR")
    & annual["year"].eq(2050)
    & annual["impact_category_id"].eq("climate_change"),
    ["background_scenario", "allocated_cc_case", "score", "allocated_cc", "unit", "asr", "compliant"],
]

## Read the cumulative climate assessment

For climate change, absolute environmental sustainability is additionally
assessed over the cumulative 2019--2100 period. The cumulative ASR divides
summed annual all-GHG GWP100 burdens by the allocated carrying capacity supplied
for that period.

The next cell reads the regional cumulative results for EUR. Global rows are
calculated by dividing the sum of regional burdens by the sum of the
corresponding allocated carrying capacities.


In [ ]:
cumulative = pd.concat(
    [pd.read_parquet(item.cumulative_gwp100_asr) for item in asr_report.results],
    ignore_index=True,
)
cumulative.loc[
    cumulative["region"].eq("EUR"),
    ["background_scenario", "allocated_cc_case", "period_start", "period_end", "score", "allocated_cc", "asr", "compliant"],
]

## Compare the selected allocated carrying capacity cases

The following cell checks annual compliance across all regions, impact
categories, and assessment years for each background and allocated carrying
capacity case. `True` indicates that every annual ASR over 2020--2100 is at
most 1. Compare these results with the cumulative climate assessment above,
which covers 2019--2100.


In [ ]:
regional_annual = annual.loc[
    annual["assessed_for_compliance"] & annual["region"].ne("global")
]
regional_annual.groupby(["background_scenario", "allocated_cc_case"])["compliant"].all()

## Record the inputs and reuse the result

Each manifest records the LCA result and allocated carrying capacity inputs
used in the assessment, together with the output file hashes. Keep it with the
tables when a result is cited or shared.

Repeating the same request reuses its completed result. Use `refresh=True`
to recalculate the selected ASR results from the completed inputs. To apply
revised allocated carrying capacities, supply them under a new
`allocated_cc_result` name.


In [ ]:
item = asr_report.results[0]
manifest = json.loads(item.manifest_path.read_text(encoding="utf-8"))
manifest["identity_payload"]